# 🔴 EcoSense — Anomaly Detection
## Multi-Method Anomaly Detection & Comparison

This notebook implements and compares 4 anomaly detection methods:

1. **Z-Score** — Statistical threshold-based
2. **IQR (Interquartile Range)** — Robust non-parametric
3. **Rolling Statistics** — Local context-aware
4. **Isolation Forest** — ML-based unsupervised

We evaluate each method against the known injected anomalies (ground truth)
and compare precision, recall, and F1 scores.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.anomaly_detector import (
    ZScoreDetector, IQRDetector, RollingStatsDetector, IsolationForestDetector,
    run_all_detectors, compare_detectors, create_ensemble_prediction
)
from src.feature_engineering import engineer_features
from src.data_generator import SENSOR_CONFIG

plt.style.use('seaborn-v0_8-darkgrid')
print('Anomaly detection modules loaded!')

In [ ]:
# Load data
water_df = pd.read_csv('../data/raw/water_quality.csv', parse_dates=['timestamp'])
air_df = pd.read_csv('../data/raw/air_quality.csv', parse_dates=['timestamp'])
anomaly_log = pd.read_csv('../data/anomalies/injected_anomalies.csv', parse_dates=['start_time', 'end_time'])

water_sensors = ['pH', 'turbidity_ntu', 'dissolved_oxygen_mg_l', 'temperature_c', 'conductivity_us_cm']
air_sensors = ['co2_ppm', 'nox_ppm', 'so2_ppm', 'pm25_ug_m3']

print(f'Data loaded. Water: {water_df.shape}, Air: {air_df.shape}')
print(f'Known anomaly events: {len(anomaly_log)}')

---
## 1. Single Sensor Deep Dive — pH

Let's demonstrate all 4 methods on a single sensor first.

In [ ]:
# Select pH sensor for detailed analysis
target_sensor = 'pH'
sensor_data = water_df[target_sensor].fillna(water_df[target_sensor].mean())
ground_truth = water_df[f'is_anomaly_{target_sensor}']

print(f'Sensor: {target_sensor}')
print(f'Total data points: {len(sensor_data):,}')
print(f'Known anomalous points: {ground_truth.sum():,} ({ground_truth.mean()*100:.2f}%)')

In [ ]:
# Prepare features for Isolation Forest
normal_ranges = {s: (c['normal_min'], c['normal_max']) 
                 for s, c in SENSOR_CONFIG['water_quality'].items()}

# Engineer features for the target sensor
feature_cols = [target_sensor]
featured_df = engineer_features(
    water_df[['timestamp', target_sensor]].copy(),
    sensor_columns=feature_cols,
    normal_ranges=normal_ranges,
    timestamp_col='timestamp'
)

# Select numeric features for Isolation Forest
if_features = featured_df.select_dtypes(include=[np.number]).drop(
    columns=[c for c in featured_df.columns if 'is_anomaly' in c], errors='ignore'
).fillna(0)

print(f'Features for Isolation Forest: {if_features.shape[1]} columns')

In [ ]:
# Run all detectors
results = run_all_detectors(
    data=sensor_data,
    features_df=if_features,
    zscore_threshold=3.0,
    iqr_factor=1.5,
    rolling_window=96,
    rolling_threshold=3.0,
    if_contamination=0.05
)

print('All detectors executed.')
for name, res in results.items():
    n_detected = res['is_anomaly'].sum()
    print(f'  {name}: {n_detected:,} anomalies detected ({n_detected/len(res)*100:.2f}%)')

In [ ]:
# Compare against ground truth
comparison = compare_detectors(results, ground_truth=ground_truth)
print('\n=== Detection Method Comparison ===')
comparison

### 1.1 Visual Comparison

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(18, 20), sharex=True)
fig.suptitle(f'Anomaly Detection Comparison — {target_sensor}', fontsize=16, fontweight='bold')

timestamps = water_df['timestamp']

# Original signal with ground truth
ax = axes[0]
ax.plot(timestamps, sensor_data, linewidth=0.4, color='steelblue', alpha=0.7)
gt_mask = ground_truth == 1
ax.scatter(timestamps[gt_mask], sensor_data[gt_mask], c='red', s=2, alpha=0.5, label='Ground Truth')
ax.set_title('Original Signal + Ground Truth Anomalies', fontweight='bold')
ax.legend(loc='upper right')

# Method results
method_colors = {'Z-Score': '#E63946', 'IQR': '#457B9D', 
                 'Rolling Stats': '#2A9D8F', 'Isolation Forest': '#F4A261'}

for idx, (method_name, result_df) in enumerate(results.items()):
    ax = axes[idx + 1]
    ax.plot(timestamps, sensor_data, linewidth=0.4, color='gray', alpha=0.4)
    
    detected_mask = result_df['is_anomaly'] == 1
    color = method_colors.get(method_name, 'orange')
    ax.scatter(timestamps[detected_mask], sensor_data[detected_mask],
              c=color, s=3, alpha=0.6, label=f'Detected ({detected_mask.sum()})')
    
    # Add rolling bands if available
    if 'upper_band' in result_df.columns:
        ax.fill_between(timestamps, result_df['lower_band'], result_df['upper_band'],
                       alpha=0.15, color=color)
    
    # Show precision/recall in title
    row = comparison[comparison['method'] == method_name].iloc[0]
    precision = row.get('precision', 'N/A')
    recall = row.get('recall', 'N/A')
    f1 = row.get('f1_score', 'N/A')
    ax.set_title(f'{method_name}  |  Precision: {precision}  |  Recall: {recall}  |  F1: {f1}',
                fontweight='bold', fontsize=10)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.savefig('../reports/anomaly_comparison_pH.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. All Sensors — Batch Detection

Run all detectors across all water and air quality sensors.

In [ ]:
all_comparisons = []

# Water quality sensors
for sensor in water_sensors:
    data = water_df[sensor].fillna(water_df[sensor].mean())
    gt = water_df.get(f'is_anomaly_{sensor}', pd.Series(0, index=water_df.index))
    
    results = run_all_detectors(data=data)
    comp = compare_detectors(results, ground_truth=gt)
    comp['sensor'] = sensor
    comp['category'] = 'Water Quality'
    all_comparisons.append(comp)

# Air quality sensors
for sensor in air_sensors:
    data = air_df[sensor].fillna(air_df[sensor].mean())
    gt = air_df.get(f'is_anomaly_{sensor}', pd.Series(0, index=air_df.index))
    
    results = run_all_detectors(data=data)
    comp = compare_detectors(results, ground_truth=gt)
    comp['sensor'] = sensor
    comp['category'] = 'Air Quality'
    all_comparisons.append(comp)

full_comparison = pd.concat(all_comparisons, ignore_index=True)
print(f'Comparison complete: {len(full_comparison)} rows')

In [ ]:
# Average performance by method
avg_performance = full_comparison.groupby('method')[['precision', 'recall', 'f1_score']].mean().round(4)
avg_performance = avg_performance.sort_values('f1_score', ascending=False)

print('=== Average Detection Performance Across All Sensors ===')
avg_performance

In [ ]:
# Heatmap: F1 score by method × sensor
pivot_f1 = full_comparison.pivot_table(values='f1_score', index='sensor', columns='method')

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='RdYlGn', 
            linewidths=1, ax=ax, vmin=0, vmax=1,
            cbar_kws={'label': 'F1 Score'})
ax.set_title('F1 Score — Method × Sensor Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Sensor')
ax.set_xlabel('Detection Method')

plt.tight_layout()
plt.savefig('../reports/f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Ensemble Voting

In [ ]:
# Ensemble on pH sensor
ph_data = water_df['pH'].fillna(water_df['pH'].mean())
ph_gt = water_df['is_anomaly_pH']

ph_results = run_all_detectors(data=ph_data)

for min_votes in [1, 2, 3]:
    ensemble = create_ensemble_prediction(ph_results, min_votes=min_votes)
    
    tp = ((ensemble == 1) & (ph_gt == 1)).sum()
    fp = ((ensemble == 1) & (ph_gt == 0)).sum()
    fn = ((ensemble == 0) & (ph_gt == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f'Ensemble (min_votes={min_votes}): '
          f'Detected={ensemble.sum()}, '
          f'Precision={precision:.4f}, '
          f'Recall={recall:.4f}, '
          f'F1={f1:.4f}')

---
## 4. Anomaly Type Analysis

In [ ]:
# Analyze which types of anomalies are detected best
if not anomaly_log.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Anomaly type distribution
    type_counts = anomaly_log['anomaly_type'].value_counts()
    colors = ['#E63946', '#457B9D', '#2A9D8F', '#F4A261']
    axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
               colors=colors[:len(type_counts)], startangle=90)
    axes[0].set_title('Anomaly Type Distribution', fontweight='bold')
    
    # Duration distribution by type
    anomaly_log.boxplot(column='duration_hours', by='anomaly_type', ax=axes[1],
                       patch_artist=True)
    axes[1].set_title('Anomaly Duration by Type', fontweight='bold')
    axes[1].set_xlabel('Anomaly Type')
    axes[1].set_ylabel('Duration (hours)')
    plt.suptitle('')  # Remove auto-title
    
    plt.tight_layout()
    plt.savefig('../reports/anomaly_types.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Anomaly timeline
    print('\n=== Anomaly Event Timeline ===')
    for _, row in anomaly_log.iterrows():
        print(f"  [{row['anomaly_type']:20s}] {row['sensor']:25s} | "
              f"{row['start_time']} → {row['end_time']} ({row['duration_hours']}h)")

In [ ]:
# Save detected anomalies for dashboard
# Using best performing method (Rolling Stats as baseline)
all_detected = []

for sensor in water_sensors:
    data = water_df[sensor].fillna(water_df[sensor].mean())
    detector = RollingStatsDetector(window=96, threshold=3.0)
    result = detector.detect(data)
    result['sensor'] = sensor
    result['category'] = 'Water Quality'
    result['timestamp'] = water_df['timestamp']
    all_detected.append(result[result['is_anomaly'] == 1])

for sensor in air_sensors:
    data = air_df[sensor].fillna(air_df[sensor].mean())
    detector = RollingStatsDetector(window=96, threshold=3.0)
    result = detector.detect(data)
    result['sensor'] = sensor
    result['category'] = 'Air Quality'
    result['timestamp'] = air_df['timestamp']
    all_detected.append(result[result['is_anomaly'] == 1])

detected_df = pd.concat(all_detected, ignore_index=True)
detected_df.to_csv('../data/anomalies/detected_anomalies.csv', index=False)

print(f'\nSaved {len(detected_df)} detected anomaly points to data/anomalies/detected_anomalies.csv')
print(f'\nDetection summary by sensor:')
print(detected_df.groupby('sensor')['is_anomaly'].count().sort_values(ascending=False))

# Save full comparison table
full_comparison.to_csv('../data/anomalies/method_comparison.csv', index=False)
print(f'\n✅ Anomaly detection complete! Proceed to notebook 04_predictive_maintenance.ipynb')